# Notebook 01: Data Exploration and Validation

## Objective
Understand the structure, quality, and content of the synthetic resume dataset before any downstream processing.
This notebook covers:
- Schema and data type validation
- Missing value analysis
- Duplicate detection
- Distribution analysis for key fields
- Binary flag analysis
- Skill vocabulary inspection
- Score distribution analysis

## Dataset
5,000 synthetic resume records with 34 columns covering candidate identity, education, experience, extracted text fields, binary experience flags, and computed scores.

## Assumptions
- The dataset is synthetic and does not represent real candidates.
- It is used exclusively for benchmarking and population analysis.
- The score columns (keyword_density_score, profile_completeness_score) are pre-computed and not re-derived here.

## Scope
Exploration and validation only. No preprocessing, modeling, or external data integration.

In [ ]:
# importing libraries
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# loading dataset
df = pd.read_csv("../Dataset/parsed_resumes.csv")
# basic shape and schema
print("Shape:", df.shape)
print()
print("Column names and dtypes:")
print(df.dtypes)
print()
print("First 3 records (text columns truncated):")

# truncate wide text columns for display
text_cols = [
    "technical_skills_raw", "tools_platforms_raw", "soft_skills_raw",
    "experience_summary", "project_summary", "key_achievements"
]
df_display = df.copy()
for col in text_cols:
    df_display[col] = df_display[col].astype(str).str[:60] + "..."

print(df_display.head(3).to_string())

Shape: (5000, 34)

Column names and dtypes:
candidate_id                               object
resume_file_name                           object
candidate_name                             object
primary_domain                             object
primary_role                               object
years_experience                            int64
highest_education                          object
education_field                            object
institution_tier                           object
technical_skills_raw                       object
tools_platforms_raw                        object
soft_skills_raw                            object
experience_summary                         object
project_summary                            object
key_achievements                           object
management_experience_flag                  int64
people_management_flag                      int64
project_management_experience_flag          int64
agile_scrum_experience_flag                 int64
client

## Section 1: Data Quality — Missing Values and Duplicates

Check for nulls across all columns and identify any duplicate records.
Duplicates are checked on candidate_id (should be unique), resume_file_name (should be unique), and full-row level.

In [3]:
# missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).query("missing_count > 0")

if missing_report.empty:
    print("No missing values found.")
else:
    print("Columns with missing values:")
    print(missing_report.to_string())

print()

# duplicate detection
dup_id = df["candidate_id"].duplicated().sum()
dup_filename = df["resume_file_name"].duplicated().sum()
dup_full = df.duplicated().sum()

print(f"Duplicate candidate_id values:    {dup_id}")
print(f"Duplicate resume_file_name values: {dup_filename}")
print(f"Fully duplicate rows:              {dup_full}")

print()

# check candidate_id format consistency
id_pattern_check = df["candidate_id"].str.match(r"^CND_\d+$")
print(f"candidate_id format violations: {(~id_pattern_check).sum()}")

Columns with missing values:
                   missing_count  missing_pct
highest_education            270         5.40
education_field              237         4.74

Duplicate candidate_id values:    0
Duplicate resume_file_name values: 0
Fully duplicate rows:              0

candidate_id format violations: 0


## Section 2: Missing Value Pattern Analysis

Investigate whether missing education values are random or concentrated in specific domains or roles.
Also check overlap between the two missing columns.

In [4]:
# identify rows with missing education values
missing_edu = df["highest_education"].isnull()
missing_field = df["education_field"].isnull()

# overlap
both_missing = (missing_edu & missing_field).sum()
only_edu_missing = (missing_edu & ~missing_field).sum()
only_field_missing = (~missing_edu & missing_field).sum()

print("Missing value overlap:")
print(f"  Both columns missing:           {both_missing}")
print(f"  Only highest_education missing: {only_edu_missing}")
print(f"  Only education_field missing:   {only_field_missing}")
print()

# domain distribution of missing highest_education
print("highest_education nulls by primary_domain:")
print(df[missing_edu]["primary_domain"].value_counts().to_string())
print()

# domain distribution of missing education_field
print("education_field nulls by primary_domain:")
print(df[missing_field]["primary_domain"].value_counts().to_string())
print()

# check if missing education correlates with years_experience
print("Years of experience — missing vs non-missing highest_education:")
print(df.groupby(missing_edu)["years_experience"].describe().round(2).to_string())

Missing value overlap:
  Both columns missing:           8
  Only highest_education missing: 262
  Only education_field missing:   229

highest_education nulls by primary_domain:
primary_domain
IT              133
Data Science     60
HR               31
Legal            29
Engineering       9
Management        8

education_field nulls by primary_domain:
primary_domain
IT              130
Data Science     45
HR               28
Legal            20
Engineering      11
Management        3

Years of experience — missing vs non-missing highest_education:
                    count  mean   std  min  25%  50%  75%   max
highest_education                                              
False              4730.0  6.72  4.02  1.0  4.0  6.0  9.0  20.0
True                270.0  6.90  4.21  1.0  4.0  6.0  9.0  20.0


## Section 3: Domain and Role Distribution

Understand how candidates are distributed across primary domains and roles.
This establishes the population composition of the benchmark dataset.

In [5]:
# domain distribution
print("Primary domain distribution:")
domain_counts = df["primary_domain"].value_counts()
domain_pct = (domain_counts / len(df) * 100).round(2)
domain_summary = pd.DataFrame({"count": domain_counts, "pct": domain_pct})
print(domain_summary.to_string())
print()

# number of unique roles overall
print(f"Total unique primary_role values: {df['primary_role'].nunique()}")
print()

# top 20 roles by frequency
print("Top 20 roles by frequency:")
print(df["primary_role"].value_counts().head(20).to_string())
print()

# roles per domain — how many distinct roles exist within each domain
print("Distinct roles per domain:")
print(df.groupby("primary_domain")["primary_role"].nunique().sort_values(ascending=False).to_string())

Primary domain distribution:
                count    pct
primary_domain              
IT               2772  55.44
Data Science     1002  20.04
HR                501  10.02
Legal             397   7.94
Engineering       189   3.78
Management        139   2.78

Total unique primary_role values: 25

Top 20 roles by frequency:
primary_role
Junior Software Engineer         490
Senior Software Engineer         485
Cloud Engineer                   478
Full Stack Developer             447
Software Engineer                442
DevOps Engineer                  430
Data Scientist                   259
Data Analyst                     255
ML Engineer                      249
Junior Data Scientist            239
HR Business Partner              140
HR Manager                       129
Talent Acquisition Specialist    116
HR Executive                     116
Corporate Lawyer                 110
Compliance Officer               103
Legal Associate                  100
Litigation Lawyer              

## Section 4: Experience and Education Distribution

Analyze years of experience, education levels, institution tiers, and education fields.
These are key inputs to ATS scoring and population benchmarking.

In [6]:
# years of experience distribution
print("Years of experience — overall statistics:")
print(df["years_experience"].describe().round(2).to_string())
print()

print("Years of experience — value counts:")
print(df["years_experience"].value_counts().sort_index().to_string())
print()

# education level distribution
print("Highest education distribution:")
edu_counts = df["highest_education"].value_counts(dropna=False)
edu_pct = (edu_counts / len(df) * 100).round(2)
print(pd.DataFrame({"count": edu_counts, "pct": edu_pct}).to_string())
print()

# institution tier distribution
print("Institution tier distribution:")
tier_counts = df["institution_tier"].value_counts(dropna=False)
tier_pct = (tier_counts / len(df) * 100).round(2)
print(pd.DataFrame({"count": tier_counts, "pct": tier_pct}).to_string())

Years of experience — overall statistics:
count    5000.00
mean        6.73
std         4.03
min         1.00
25%         4.00
50%         6.00
75%         9.00
max        20.00

Years of experience — value counts:
years_experience
1     257
2     271
3     643
4     629
5     408
6     438
7     413
8     447
9     448
10    513
11     59
12     54
13     63
14     55
15     55
16     57
17     52
18     45
19     50
20     43

Highest education distribution:
                   count    pct
highest_education              
LLM                 1216  24.32
Masters             1202  24.04
MBA                 1185  23.70
Bachelors           1127  22.54
NaN                  270   5.40

Institution tier distribution:
                  count    pct
institution_tier              
Tier-3             1714  34.28
Tier-1             1667  33.34
Tier-2             1619  32.38


## Section 5: Education Field and LLM Credential Investigation

Investigate the education_field vocabulary and confirm the domain context
of LLM credential holders. Determine whether LLM maps cleanly to a single domain.

In [7]:
# education field distribution
print("Education field distribution:")
field_counts = df["education_field"].value_counts(dropna=False)
field_pct = (field_counts / len(df) * 100).round(2)
print(pd.DataFrame({"count": field_counts, "pct": field_pct}).to_string())
print()

# cross-tab: highest_education vs primary_domain for LLM credential
print("LLM credential holders by domain:")
llm_mask = df["highest_education"] == "LLM"
print(df[llm_mask]["primary_domain"].value_counts().to_string())
print()

# cross-tab: highest_education vs primary_domain (full picture)
print("Education level by domain (counts):")
edu_domain = pd.crosstab(df["primary_domain"], df["highest_education"], dropna=False)
print(edu_domain.to_string())

Education field distribution:
                         count    pct
education_field                      
Information Technology    1331  26.62
Computer Science          1311  26.22
Law                        377   7.54
Data Science               340   6.80
Mathematics                316   6.32
Statistics                 301   6.02
Human Resources            243   4.86
NaN                        237   4.74
Psychology                 230   4.60
Mechanical Engineering     178   3.56
Management                  77   1.54
Business Administration     59   1.18

LLM credential holders by domain:
primary_domain
IT              667
Data Science    252
HR              121
Legal           102
Engineering      45
Management       29

Education level by domain (counts):
highest_education  Bachelors  LLM  MBA  Masters  NaN
primary_domain                                      
Data Science             223  252  248      219   60
Engineering               43   45   44       48    9
HR                 

## Section 6: Binary Experience Flag Analysis

Examine the 15 binary experience flags.
Check value distributions, flag co-occurrence patterns, and domain-level flag rates.
These flags are direct inputs to ATS scoring and role matching.

In [8]:
# define flag columns
flag_cols = [
    "management_experience_flag",
    "people_management_flag",
    "project_management_experience_flag",
    "agile_scrum_experience_flag",
    "client_facing_experience_flag",
    "delivery_lead_experience_flag",
    "cloud_experience_flag",
    "ml_experience_flag",
    "compliance_experience_flag",
    "enterprise_systems_experience_flag",
    "offshore_onsite_model_experience_flag",
    "multi_vendor_coordination_flag",
    "process_compliance_experience_flag",
    "documentation_heavy_role_flag",
    "mentoring_experience_flag",
    "stakeholder_management_experience_flag"
]

# overall flag rates
flag_rates = df[flag_cols].mean().round(3).sort_values(ascending=False)
print("Flag activation rates (proportion of candidates with flag = 1):")
print(flag_rates.to_string())
print()

# check for any non-binary values in flag columns
print("Non-binary values in flag columns:")
for col in flag_cols:
    unique_vals = df[col].unique()
    if not set(unique_vals).issubset({0, 1}):
        print(f"  {col}: {unique_vals}")
print("  (none)" if all(set(df[col].unique()).issubset({0, 1}) for col in flag_cols) else "")
print()

# flag rates by domain
print("Flag rates by primary_domain:")
print(df.groupby("primary_domain")[flag_cols].mean().round(3).to_string())

Flag activation rates (proportion of candidates with flag = 1):
offshore_onsite_model_experience_flag     0.894
agile_scrum_experience_flag               0.783
process_compliance_experience_flag        0.766
client_facing_experience_flag             0.766
documentation_heavy_role_flag             0.662
stakeholder_management_experience_flag    0.640
project_management_experience_flag        0.640
enterprise_systems_experience_flag        0.558
mentoring_experience_flag                 0.558
multi_vendor_coordination_flag            0.471
management_experience_flag                0.388
delivery_lead_experience_flag             0.299
cloud_experience_flag                     0.278
people_management_flag                    0.209
ml_experience_flag                        0.200
compliance_experience_flag                0.180

Non-binary values in flag columns:
  (none)

Flag rates by primary_domain:
                management_experience_flag  people_management_flag  project_management_exper

## Section 7: Computed Score Distributions

Analyze keyword_density_score and profile_completeness_score.
These are pre-computed synthetic scores that will anchor the ATS scoring benchmarks.
Check distributions, ranges, and whether scores vary meaningfully by domain and experience level.

In [9]:
# overall score statistics
print("keyword_density_score statistics:")
print(df["keyword_density_score"].describe().round(3).to_string())
print()

print("profile_completeness_score statistics:")
print(df["profile_completeness_score"].describe().round(3).to_string())
print()

# check for out-of-range values (expecting 0.0 to 1.0)
kd_out = df[(df["keyword_density_score"] < 0) | (df["keyword_density_score"] > 1)]
pc_out = df[(df["profile_completeness_score"] < 0) | (df["profile_completeness_score"] > 1)]
print(f"keyword_density_score out of [0,1] range:      {len(kd_out)}")
print(f"profile_completeness_score out of [0,1] range: {len(pc_out)}")
print()

# score distributions by domain
print("keyword_density_score mean by domain:")
print(df.groupby("primary_domain")["keyword_density_score"].mean().round(3).sort_values(ascending=False).to_string())
print()

print("profile_completeness_score mean by domain:")
print(df.groupby("primary_domain")["profile_completeness_score"].mean().round(3).sort_values(ascending=False).to_string())
print()

# correlation between the two scores
corr = df[["keyword_density_score", "profile_completeness_score"]].corr().iloc[0, 1]
print(f"Correlation between keyword_density and profile_completeness: {corr:.3f}")
print()

# correlation with years_experience
print("Correlation with years_experience:")
print(df[["years_experience", "keyword_density_score", "profile_completeness_score"]].corr().round(3).to_string())

keyword_density_score statistics:
count    5000.000
mean        0.578
std         0.187
min         0.250
25%         0.420
50%         0.580
75%         0.740
max         0.900

profile_completeness_score statistics:
count    5000.000
mean        0.752
std         0.115
min         0.550
25%         0.650
50%         0.750
75%         0.850
max         0.950

keyword_density_score out of [0,1] range:      0
profile_completeness_score out of [0,1] range: 0

keyword_density_score mean by domain:
primary_domain
Engineering     0.587
Legal           0.587
HR              0.578
Management      0.577
Data Science    0.576
IT              0.576

profile_completeness_score mean by domain:
primary_domain
Management      0.760
Legal           0.758
HR              0.754
IT              0.751
Data Science    0.749
Engineering     0.744

Correlation between keyword_density and profile_completeness: -0.017

Correlation with years_experience:
                            years_experience  keyword_de

## Section 8: Text Field Quality Analysis

Inspect the raw text fields: technical_skills_raw, tools_platforms_raw, soft_skills_raw,
experience_summary, project_summary, and key_achievements.
Check for empty strings, placeholder text, length distributions, and vocabulary patterns.
These fields are the primary inputs to NLP processing and semantic matching.

In [10]:
# text columns to inspect
text_cols = {
    "technical_skills_raw": "skills",
    "tools_platforms_raw": "tools",
    "soft_skills_raw": "soft",
    "experience_summary": "exp_summary",
    "project_summary": "proj_summary",
    "key_achievements": "achievements"
}

# compute length stats and check for empty or placeholder content
print("Text field length statistics (character count):")
length_stats = {}
for col in text_cols:
    lengths = df[col].astype(str).str.len()
    length_stats[col] = lengths

length_df = pd.DataFrame(length_stats).describe().round(1)
print(length_df.to_string())
print()

# check for empty strings or whitespace-only values
print("Empty or whitespace-only values per text field:")
for col in text_cols:
    empty = df[col].astype(str).str.strip().eq("").sum()
    nan_like = df[col].astype(str).str.lower().isin(["nan", "none", "null", "n/a", ""]).sum()
    print(f"  {col}: empty={empty}, nan-like={nan_like}")
print()

# sample 3 values from each field to assess content quality
print("Sample values from technical_skills_raw:")
print(df["technical_skills_raw"].dropna().sample(3, random_state=42).to_string())
print()

print("Sample values from experience_summary:")
print(df["experience_summary"].dropna().sample(3, random_state=42).to_string())

Text field length statistics (character count):
       technical_skills_raw  tools_platforms_raw  soft_skills_raw  experience_summary  project_summary  key_achievements
count                5000.0               5000.0           5000.0              5000.0           5000.0            5000.0
mean                   34.6                 13.7             26.3               131.0            105.7              97.4
std                    12.0                  3.8              3.4                 8.2              9.2               4.6
min                    19.0                  8.0             20.0               123.0             91.0              90.0
25%                    25.0                 10.0             25.0               123.0             94.0              91.0
50%                    30.0                 15.0             25.0               132.0            109.0              99.0
75%                    43.0                 16.0             30.0               138.0            112.0   

## Section 9: Skill Vocabulary Analysis

Extract and analyze the unique skill tokens from technical_skills_raw and tools_platforms_raw.
Understand vocabulary size, frequency distribution, and domain-level skill patterns.
This informs ESCO normalization scope and skill extraction design.

In [11]:
# parse comma-separated skill strings into token lists
def parse_tokens(series):
    tokens = []
    for val in series.dropna():
        parts = [t.strip() for t in str(val).split(",") if t.strip()]
        tokens.extend(parts)
    return tokens

tech_tokens = parse_tokens(df["technical_skills_raw"])
tool_tokens = parse_tokens(df["tools_platforms_raw"])

from collections import Counter

tech_counts = Counter(tech_tokens)
tool_counts = Counter(tool_tokens)

print(f"Total technical skill mentions:      {len(tech_tokens)}")
print(f"Unique technical skill tokens:       {len(tech_counts)}")
print()
print(f"Total tools/platforms mentions:      {len(tool_tokens)}")
print(f"Unique tools/platforms tokens:       {len(tool_counts)}")
print()

print("Top 30 technical skills by frequency:")
for skill, count in tech_counts.most_common(30):
    print(f"  {skill:<40} {count}")
print()

print("Top 20 tools and platforms by frequency:")
for tool, count in tool_counts.most_common(20):
    print(f"  {tool:<40} {count}")

Total technical skill mentions:      20000
Unique technical skill tokens:       212

Total tools/platforms mentions:      10000
Unique tools/platforms tokens:       6

Top 30 technical skills by frequency:
  SQL                                      1923
  Python                                   1796
  AWS                                      1388
  Git                                      1382
  Docker                                   1298
  Linux                                    1293
  Kubernetes                               1271
  Java                                     1251
  NLP                                      561
  TensorFlow                               554
  Power BI                                 546
  Pandas                                   526
  Machine Learning                         522
  HRMS                                     383
  Recruitment                              381
  HR Analytics                             381
  Contract Drafting               

## Section 10: Resume Length Distribution

Check resume_length_words distribution and its relationship with experience level.
This field is used as a proxy for resume completeness in ATS scoring.

In [12]:
# resume length statistics
print("resume_length_words statistics:")
print(df["resume_length_words"].describe().round(2).to_string())
print()

# value range check
print(f"Min word count: {df['resume_length_words'].min()}")
print(f"Max word count: {df['resume_length_words'].max()}")
print()

# distribution by experience band
bins = [0, 3, 6, 10, 20]
labels = ["Junior (1-3)", "Mid (4-6)", "Senior (7-10)", "Expert (11-20)"]
df["exp_band"] = pd.cut(df["years_experience"], bins=bins, labels=labels)

print("Resume length by experience band:")
print(df.groupby("exp_band", observed=True)["resume_length_words"].describe().round(2).to_string())
print()

# correlation with scores
print("Correlations with resume_length_words:")
corr_cols = ["resume_length_words", "keyword_density_score", "profile_completeness_score", "years_experience"]
print(df[corr_cols].corr().round(3).to_string())

# drop temporary column
df.drop(columns=["exp_band"], inplace=True)

resume_length_words statistics:
count    5000.00
mean       54.85
std         2.89
min        48.00
25%        53.00
50%        55.00
75%        57.00
max        63.00

Min word count: 48
Max word count: 63

Resume length by experience band:
                 count   mean   std   min   25%   50%   75%   max
exp_band                                                         
Junior (1-3)    1171.0  54.99  2.83  48.0  53.0  55.0  57.0  63.0
Mid (4-6)       1475.0  54.92  2.92  48.0  53.0  55.0  57.0  62.0
Senior (7-10)   1821.0  54.87  2.87  48.0  53.0  55.0  57.0  63.0
Expert (11-20)   533.0  54.27  2.94  48.0  52.0  54.0  56.0  62.0

Correlations with resume_length_words:
                            resume_length_words  keyword_density_score  profile_completeness_score  years_experience
resume_length_words                       1.000                  0.009                      -0.014            -0.062
keyword_density_score                     0.009                  1.000                  

## Notebook 01 Findings Summary

### Dataset Overview
- 5,000 records, 34 columns, no duplicate records, no candidate_id format violations.
- 6 domains, 25 roles, clean controlled vocabularies throughout.
- Dataset is suitable for population benchmarking. It is not suitable for NLP pipeline validation.

### Data Quality Issues

**Missing values**
- highest_education: 270 nulls (5.4%)
- education_field: 237 nulls (4.74%)
- Only 8 records missing both simultaneously — columns were generated independently.
- Missing values are proportional to domain size, consistent with missing at random.
- Preprocessing decision: impute with a sentinel value such as "Unknown" rather than dropping records.

**Education label vocabulary**
- highest_education contains four values: Bachelors, Masters, MBA, LLM.
- LLM is used as a generic postgraduate tier, not as a law-specific credential.
- Distribution is uniform across all domains — LLM was assigned randomly like the other tiers.
- Preprocessing decision: map LLM to Masters tier for ATS scoring purposes.

### Population Composition
- IT dominates at 55.4%, Data Science at 20%, HR 10%, Legal 8%, Engineering 4%, Management 3%.
- Management (139 records) and Engineering (189 records) will produce unstable percentile estimates.
- Experience distribution is bimodal: dense at years 1-10, thin tail at years 11-20.
- Percentile benchmarks for candidates with 11+ years of experience will be unreliable.
- Institution tier is artificially uniform at roughly 33% each — a synthetic artifact.

### Binary Flag Limitations
- Several flags are domain-level constants, not candidate-level signals.
- agile_scrum_experience_flag: always 1 for Data Science, IT, Management; always 0 for Engineering, HR, Legal.
- ml_experience_flag: always 1 for Data Science; always 0 for all other domains.
- compliance_experience_flag: always 1 for HR and Legal; always 0 for all other domains.
- documentation_heavy_role_flag: always 0 for Data Science, Engineering, HR; always 1 for IT, Legal, Management.
- cloud_experience_flag: only varies within IT; zero for all other domains.
- These flags are useful for cross-domain role matching but useless for within-domain benchmarking.
- offshore_onsite_model_experience_flag activates at 89.4% overall — near-constant, weak discriminator globally.

### Pre-computed Score Limitations
- keyword_density_score and profile_completeness_score are statistically independent of each other,
  of years_experience, and of domain. Correlation between them is -0.017.
- Domain means differ by at most 0.016 — scores are effectively random within fixed ranges.
- These scores cannot serve as ground truth for ATS scoring.
- They can only be used as population distribution references for percentile benchmarking.

### Text Field Limitations
- All text fields are templated with very low variance.
- experience_summary ranges from 123 to 149 characters across 5,000 records — 26 character spread.
- key_achievements is hard-capped at 102 characters.
- tools_platforms_raw contains only 6 unique tool names across 10,000 mentions.
- Semantic similarity scores computed on these fields will cluster tightly and will not
  differentiate candidates meaningfully.
- This dataset cannot be used to validate NLP pipeline quality. Real resume documents are required.

### resume_length_words
- Range is 48 to 63 with std of 2.89. This is not a word count.
- Field appears to count template slots or skill tokens, not actual words.
- Zero correlation with experience, domain, or scores.
- Cannot be used as a resume completeness proxy. Exclude from ATS scoring feature set.

### Skill Vocabulary
- 212 unique technical skill tokens, clean and domain-segmented with no noise.
- 6 unique tool tokens: GCP, ServiceNow, Azure, AWS, Confluence, Jira.
- AWS appears in both technical_skills_raw and tools_platforms_raw — deduplicate when building skill profiles.
- Vocabulary will map cleanly to ESCO with minimal fuzzy matching needed.

### Decisions Made
1. LLM maps to Masters tier in preprocessing.
2. Missing education values imputed with sentinel "Unknown".
3. resume_length_words excluded from ATS scoring feature set.
4. Pre-computed scores used only for benchmarking percentile reference, not as ground truth.
5. Constant flags within domains noted — scoring logic must account for zero within-domain variance.
6. Deduplicate skills across technical_skills_raw and tools_platforms_raw when building candidate profiles.

### Decisions Deferred
1. Exact imputation strategy for education_field nulls (depends on preprocessing design).
2. Handling of thin benchmark populations for Management and Engineering domains.
3. Percentile handling for the 11-20 year experience tail.